In [3]:
import numpy as np
import pandas as pd
from gtda.time_series import SlidingWindow, TakensEmbedding
from gtda.homology import VietorisRipsPersistence
from gtda.diagrams import PersistenceEntropy, Amplitude
from tqdm import tqdm
import json

## Opening data

In [4]:
file_path = "../data/raw/dengue.csv.gz"
data_casos = pd.read_csv(file_path)
display(data_casos.head())
print(f"Columns: {data_casos.columns}")

,geocode,date,casos,epiweek,uf,macroregional_geocode,regional_geocode,uf_code,target_city,train_1,target_1,train_2,target_2,train_3,target_3,train_4,target_4,disease
0,3301900,2010-01-03,5,201001,RJ,3312,33006,33,False,True,False,True,False,True,False,True,False,dengue
1,1504505,2010-01-03,0,201001,PA,1512,15014,15,False,True,False,True,False,True,False,True,False,dengue
2,1503077,2010-01-03,0,201001,PA,1511,15008,15,False,True,False,True,False,True,False,True,False,dengue
3,2106409,2010-01-03,0,201001,MA,2110,21006,21,False,True,False,True,False,True,False,True,False,dengue
4,3135704,2010-01-03,0,201001,MG,3103,31024,31,False,True,False,True,False,True,False,True,False,dengue


Columns: Index(['geocode', 'date', 'casos', 'epiweek', 'uf', 'macroregional_geocode',
       'regional_geocode', 'uf_code', 'target_city', 'train_1', 'target_1',
       'train_2', 'target_2', 'train_3', 'target_3', 'train_4', 'target_4',
       'disease'],
      dtype='str')


In [5]:
file_path = "../data/raw/climate.csv.gz"
data_climate = pd.read_csv(file_path)
display(data_climate.head())
print(f"Columns: {data_climate.columns}")

,date,epiweek,geocode,temp_min,temp_med,temp_max,precip_min,precip_med,precip_max,pressure_min,pressure_med,pressure_max,rel_humid_min,rel_humid_med,rel_humid_max,thermal_range,rainy_days
0,1999-12-26,199952,2700102,20.225500,25.611700,31.668000,0.0000,0.0218,0.1679,0.953100,0.955200,0.956600,37.954800,67.179900,93.839800,11.442500,1
1,2000-01-02,200001,2700102,21.935329,25.499886,30.494000,0.0132,2.6425,10.4447,0.954014,0.955986,0.957686,44.407771,70.016357,90.559814,8.558671,7
2,2000-01-09,200002,2700102,21.285600,25.022900,30.137100,1.3862,9.1152,23.0169,0.954771,0.956700,0.958186,48.266157,72.211186,92.909529,8.851500,7
3,2000-01-16,200003,2700102,22.731871,26.944043,32.038900,3.4326,6.7589,13.1685,0.954700,0.956557,0.958386,42.750943,65.206800,87.661314,9.307029,7
4,2000-01-23,200004,2700102,22.569886,27.036471,32.426229,2.6215,5.7635,18.1966,0.955114,0.957214,0.959143,39.626114,66.100400,90.010671,9.856343,6


Columns: Index(['date', 'epiweek', 'geocode', 'temp_min', 'temp_med', 'temp_max',
       'precip_min', 'precip_med', 'precip_max', 'pressure_min',
       'pressure_med', 'pressure_max', 'rel_humid_min', 'rel_humid_med',
       'rel_humid_max', 'thermal_range', 'rainy_days'],
      dtype='str')


In [6]:
file_path = "../data/raw/environ_vars.csv.gz"
data_environ = pd.read_csv(file_path)
display(data_environ.head())
print(f"Columns: {data_environ.columns}")

,geocode,uf_code,koppen,biome
0,1100015,11,Am,Amazônia
1,1100023,11,Am,Amazônia
2,1100031,11,Am,Amazônia
3,1100049,11,Am,Amazônia
4,1100056,11,Am,Amazônia


Columns: Index(['geocode', 'uf_code', 'koppen', 'biome'], dtype='str')


In [7]:
file_path = "../data/raw/forecasting_climate.csv.gz"
data_forecast_climate = pd.read_csv(file_path)
display(data_forecast_climate.head())
print(f"Columns: {data_forecast_climate.columns}")

,geocode,reference_month,forecast_months_ahead,temp_med,umid_med,precip_tot
0,1100015,2010-01-01,1,25.452503,87.700993,0.000117
1,1100015,2010-01-01,2,25.591567,87.565880,0.000112
2,1100015,2010-01-01,3,25.499011,87.660270,0.000101
3,1100015,2010-01-01,4,25.057325,86.359595,0.000058
4,1100015,2010-01-01,5,24.504658,81.563863,0.000023


Columns: Index(['geocode', 'reference_month', 'forecast_months_ahead', 'temp_med',
       'umid_med', 'precip_tot'],
      dtype='str')


In [8]:
file_path = "../data/raw/ocean_climate_oscillations.csv.gz"
data_ocean = pd.read_csv(file_path)
display(data_ocean.head())
print(f"Columns: {data_ocean.columns}")

,date,enso,iod,pdo
0,1993-01-04,1.169212,-0.467861,0.866203
1,1993-01-12,1.164812,-0.070165,1.601905
2,1993-01-19,1.041243,-0.109995,1.287732
3,1993-01-26,1.002498,-0.382814,1.384840
4,1993-02-01,1.112526,-0.639707,2.332959


Columns: Index(['date', 'enso', 'iod', 'pdo'], dtype='str')


In [9]:
file_path = "../data/raw/datasus_population_2001_2025.csv.gz"
data_pop = pd.read_csv(file_path)
display(data_pop.head())
print(f"Columns: {data_pop.columns}")

,geocode,year,population
0,1100015,2001,26553
1,3540408,2001,4496
2,3540309,2001,2567
3,3540259,2001,3637
4,3540200,2001,31595


Columns: Index(['geocode', 'year', 'population'], dtype='str')


In [10]:
file_path = "../data/raw/map_regional_health.csv"
data_reg_health = pd.read_csv(file_path)
display(data_reg_health.head())
print(f"Columns: {data_reg_health.columns}")

,macroregion_code,macroregion_name,uf_code,uf,uf_name,macroregional_geocode,macroregional_name,regional_geocode,regional_name,geocode,geocode_name
0,1,Norte,12,AC,Acre,1201,MACRO UNICA - AC,12002,BAIXO ACRE E PURUS,1200013,AC - ACRELANDIA
1,1,Norte,12,AC,Acre,1201,MACRO UNICA - AC,12001,ALTO ACRE,1200054,AC - ASSIS BRASIL
2,1,Norte,12,AC,Acre,1201,MACRO UNICA - AC,12001,ALTO ACRE,1200104,AC - BRASILEIA
3,1,Norte,12,AC,Acre,1201,MACRO UNICA - AC,12002,BAIXO ACRE E PURUS,1200138,AC - BUJARI
4,1,Norte,12,AC,Acre,1201,MACRO UNICA - AC,12002,BAIXO ACRE E PURUS,1200179,AC - CAPIXABA


Columns: Index(['macroregion_code', 'macroregion_name', 'uf_code', 'uf', 'uf_name',
       'macroregional_geocode', 'macroregional_name', 'regional_geocode',
       'regional_name', 'geocode', 'geocode_name'],
      dtype='str')


In [11]:
file_path = "../data/processed/episcanner_results_2026.csv"
data_episcanner = pd.read_csv(file_path)
display(data_episcanner.head())
print(f"Columns: {data_episcanner.columns}")

,geocode,season_end_year,disease,total_cases_L1,peak_week,R0,beta,gamma,duration_weeks,rmse,success
0,1100015,2012,dengue,37.113773,201224,1.392005,0.417601,0.300000,50,1.075852,True
1,1100015,2013,dengue,165.198386,201305,2.193093,0.657928,0.300000,19,0.980334,True
2,1100015,2016,dengue,566.030215,201604,2.238394,0.738670,0.330000,17,5.228477,True
3,1100015,2019,dengue,125.644570,201924,2.133822,0.640147,0.300000,20,17.630583,True
4,1100015,2020,dengue,68.862082,201947,2.276911,0.751046,0.329853,13,0.131021,True


Columns: Index(['geocode', 'season_end_year', 'disease', 'total_cases_L1', 'peak_week',
       'R0', 'beta', 'gamma', 'duration_weeks', 'rmse', 'success'],
      dtype='str')


In [12]:
file_path = "../data/raw/chikungunya.csv.gz"
data_casos_chik = pd.read_csv(file_path)
display(data_casos_chik.head())
print(f"Columns: {data_casos_chik.columns}")

,geocode,date,casos,epiweek,uf,macroregional_geocode,regional_geocode,uf_code,target_city,train_1,target_1,train_2,target_2,train_3,target_3,train_4,target_4,disease
0,1200013,2013-12-29,0,201401,AC,1201,12002,12,False,True,False,True,False,True,False,True,False,chikungunya
1,4312955,2013-12-29,0,201401,RS,4311,43020,43,False,True,False,True,False,True,False,True,False,chikungunya
2,4312906,2013-12-29,0,201401,RS,4310,43025,43,False,True,False,True,False,True,False,True,False,chikungunya
3,2924652,2013-12-29,0,201401,BA,2917,29006,29,False,True,False,True,False,True,False,True,False,chikungunya
4,4312807,2013-12-29,0,201401,RS,4310,43025,43,False,True,False,True,False,True,False,True,False,chikungunya


Columns: Index(['geocode', 'date', 'casos', 'epiweek', 'uf', 'macroregional_geocode',
       'regional_geocode', 'uf_code', 'target_city', 'train_1', 'target_1',
       'train_2', 'target_2', 'train_3', 'target_3', 'train_4', 'target_4',
       'disease'],
      dtype='str')


## Pre-processing

In [13]:
def calculate_rolling_tda(df, target_col='casos', group_col='geocode', window_size=53, stride=1):
    """
    Compute topological features using a sliding window.
    """
    print(f"BEGGINING TDA EXTRACTION (Window={window_size}, Column='{target_col}')...")

    TE = TakensEmbedding(dimension=3, time_delay=1)
    VR = VietorisRipsPersistence(homology_dimensions=[0, 1])
    PE = PersistenceEntropy()
    AMP = Amplitude(metric='wasserstein')

    tda_results = []

    unique_geocodes = df[group_col].unique()

    for geo in tqdm(unique_geocodes, desc="Processando Cidades"):
        city_data = df[df[group_col] == geo].sort_values('time_idx')
        series = city_data[target_col].values

        if len(series) < window_size:
            continue

        SW = SlidingWindow(size=window_size, stride=stride)
        windows = SW.fit_transform(series)

        try:
            point_clouds = TE.fit_transform(windows)

            diagrams = VR.fit_transform(point_clouds)

            entropy = PE.fit_transform(diagrams)
            amplitude = AMP.fit_transform(diagrams)

        except Exception as e:
            print(f"Erro na cidade {geo}: {e}")
            continue

        valid_indices = city_data.index[window_size - 1 :]

        df_city_tda = pd.DataFrame(index=valid_indices)
        df_city_tda[group_col] = geo

        df_city_tda['tda_entropy_H0'] = entropy[:, 0]
        df_city_tda['tda_entropy_H1'] = entropy[:, 1]
        df_city_tda['tda_amplitude_H0'] = amplitude[:, 0]
        df_city_tda['tda_amplitude_H1'] = amplitude[:, 1]

        tda_results.append(df_city_tda)

    if not tda_results:
        print("No results found.")
        return df

    df_tda_final = pd.concat(tda_results)

    print("Merging on main df")
    df_merged = df.merge(
        df_tda_final,
        left_index=True,
        right_index=True,
        how='left',
        suffixes=('', '_drop')
    )

    cols_to_drop = [c for c in df_merged.columns if '_drop' in c]
    df_merged.drop(columns=cols_to_drop, inplace=True)

    tda_cols = ['tda_entropy_H0', 'tda_entropy_H1', 'tda_amplitude_H0', 'tda_amplitude_H1']
    df_merged[tda_cols] = df_merged[tda_cols].fillna(0)

    print("Finished")
    return df_merged

In [17]:
def preprocess_for_tft(
    df_casos,
    df_climate,
    df_environ,
    df_forecast_climate,
    df_ocean,
    df_pop,
    df_reg_health,
    df_episcanner
):
    """
    Process all data for TFT training
    """

    print("Beggining Pre-processing")

    df = df_casos.copy()

    df['date'] = pd.to_datetime(df['date'])
    df_climate['date'] = pd.to_datetime(df_climate['date'])
    df_ocean['date'] = pd.to_datetime(df_ocean['date'])

    df = df.sort_values(['geocode', 'date']).reset_index(drop=True)

    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    min_date = df['date'].min()
    df['time_idx'] = ((df['date'] - min_date).dt.days / 7).astype(int)

    if 'epiweek' in df.columns:
        df['week_of_year'] = df['epiweek'].astype(str).str[-2:].astype(int)
    else:
        df['week_of_year'] = df['date'].dt.isocalendar().week.astype(int)

    df['week_cycle'] = df['week_of_year'].apply(lambda x: x - 40 if x >= 41 else x + 12)

    df['sin_week_cycle'] = np.sin(2 * np.pi * df['week_cycle'] / 52)
    df['cos_week_cycle'] = np.cos(2 * np.pi * df['week_cycle'] / 52)

    print("Integrating Climate")

    cols_clima = [c for c in df_climate.columns if c not in ['epiweek']]
    df = pd.merge(df, df_climate[cols_clima], on=['geocode', 'date'], how='left')

    present_climate_cols = [c for c in df_climate.columns if c in df.columns and c not in ['geocode', 'date']]
    df[present_climate_cols] = df.groupby('geocode')[present_climate_cols].ffill()

    print("Integrating Ocean Climate")

    df = pd.merge(df, df_ocean, on='date', how='left')
    df[['enso', 'iod', 'pdo']] = df[['enso', 'iod', 'pdo']].ffill()

    print("Integrating Forecasting")

    df_fc = df_forecast_climate.copy()
    df_fc['reference_month'] = pd.to_datetime(df_fc['reference_month'], format='mixed')
    df_fc['valid_date'] = df_fc.apply(
        lambda x: x['reference_month'] + pd.DateOffset(months=int(x['forecast_months_ahead'])),
        axis=1
    )

    df_fc['year'] = df_fc['valid_date'].dt.year
    df_fc['month'] = df_fc['valid_date'].dt.month

    rename_dict = {
        'temp_med': 'forecast_temp_med',
        'umid_med': 'forecast_umid_med',
        'precip_tot': 'forecast_precip_tot'
    }
    df_fc = df_fc.rename(columns=rename_dict)

    cols_to_use = ['geocode', 'year', 'month', 'forecast_temp_med', 'forecast_umid_med', 'forecast_precip_tot']
    df_fc_clean = df_fc[cols_to_use].groupby(['geocode', 'year', 'month']).mean().reset_index()

    df = pd.merge(df, df_fc_clean, on=['geocode', 'year', 'month'], how='left')

    for col in rename_dict.values():
        df[col] = df[col].ffill()

    print("Integrating Population")
    df = pd.merge(df, df_pop, on=['geocode', 'year'], how='left')

    df['log_pop'] = np.log1p(df['population'])
    df['log_pop'] = df.groupby('geocode')['log_pop'].ffill()

    print("Integrating Environmental Variables")

    if 'uf_code' in df.columns and 'uf_code' in df_environ.columns:
        df = df.drop(columns=['uf_code'])

    df = pd.merge(df, df_environ, on='geocode', how='left')

    cols_reg = ['geocode', 'macroregion_name', 'regional_name']
    cols_reg_exist = [c for c in cols_reg if c in df_reg_health.columns]
    df = pd.merge(df, df_reg_health[cols_reg_exist], on='geocode', how='left')

    print("Integrating Episcanner Data")

    target_cols = ['geocode', 'year', 'R0', 'peak_week', 'total_cases', 'alpha', 'beta']
    df_epi_targets = df_episcanner[target_cols].copy()

    df_epi_targets['log_total_cases'] = np.log1p(df_epi_targets['total_cases'])

    df = pd.merge(df, df_epi_targets, on=['geocode', 'year'], how='left')

    print(f"Null: {len(df)}")
    df = df.dropna(subset=['R0'])
    print(f"Complete: {len(df)}")

    df['casos'] = df['casos'].fillna(0)
    df['incidence'] = (df['casos'] / df['population']) * 100000
    df['incidence'] = df['incidence'].fillna(0)

    print("Calculating TDA Features")
    df = calculate_rolling_tda(
        df,
        target_col='incidence',
        group_col='geocode'
    )

    known_reals = [
        "time_idx",
        "week_cycle",
        "sin_week_cycle",
        "cos_week_cycle",
        "log_pop",
        "forecast_temp_med",
        "forecast_umid_med",
        "forecast_precip_tot"
    ]

    tda_features = ['tda_entropy_H0', 'tda_entropy_H1', 'tda_amplitude_H0', 'tda_amplitude_H1']

    unknown_reals = [
        "casos",
        "incidence",
        "temp_med",
        "precip_med",
        "rel_humid_med",
        "enso",
        "iod"
    ] + tda_features

    static_cats = ["koppen", "biome", "macroregion_name"]

    targets = ["R0", "peak_week", "log_total_cases", "alpha", "beta"]

    print("Finished!")
    return df, known_reals, unknown_reals, static_cats, targets

In [18]:
df_final, known, unknown, statics, targets = preprocess_for_tft(
    data_casos, data_climate, data_environ, data_forecast_climate,
    data_ocean, data_pop, data_reg_health, data_episcanner
)

Beggining Pre-processing
Integrating Climate
Integrating Ocean Climate
Integrating Forecasting
Integrating Population
Integrating Environmental Variables
Integrating Episcanner Data


KeyError: "['year', 'total_cases', 'alpha'] not in index"

## Saving

In [ ]:
def save_optimized_dataset(df, filepath):
    """
    Optimize dataset memory usage and save to parquet.
    """
    float_cols = df.select_dtypes(include=['float64']).columns
    df[float_cols] = df[float_cols].astype('float32')

    int_cols = df.select_dtypes(include=['int64', 'int']).columns
    for col in int_cols:
        c_min = df[col].min()
        c_max = df[col].max()

        if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
            df[col] = df[col].astype(np.int8)
        elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
            df[col] = df[col].astype(np.int16)
        elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
            df[col] = df[col].astype(np.int32)

    categorical_candidates = [
        'uf', 'biome', 'koppen', 'macroregion_name', 'regional_name',
        'month', 'geocode_name'
    ]

    for col in categorical_candidates:
        if col in df.columns:
            if df[col].nunique() / len(df) < 0.5:
                df[col] = df[col].astype('category')

    print("Saving compressed parquet")
    df.to_parquet(
        filepath,
        compression='zstd',
        index=False,
        engine='pyarrow'
    )
    print(f"Saved to: {filepath}")

In [ ]:
def save_tft_config(known, unknown, statics, targets, filepath="../data/processed/tft_config.json"):
    """
    Save a JSON config file with the dataset's metadata for TFT.
    """
    config = {
        "time_varying_known_reals": known,
        "time_varying_unknown_reals": unknown,
        "static_categoricals": statics,
        "targets": targets,
        "static_reals": ["num_neighbors"]
    }

    with open(filepath, 'w') as f:
        json.dump(config, f, indent=4)

    print(f"Saved to: {filepath}")

In [ ]:
save_optimized_dataset(df_final, "../data/processed/dataset_tft_completo.parquet")

In [ ]:
save_tft_config(known, unknown, statics, targets, "../data/processed/tft_config.json")

## Graph Embedding

In [ ]:
from src.graph_embedding import generate_graph_embeddings

In [ ]:
# No seu Jupyter Notebook:
df_graph, cols = generate_graph_embeddings(
    edges_path="../data/processed/adjacencia_edges.csv",
    output_path="../data/processed/graph_embeddings.csv"
)